# Gemini 비디오 질의응답


> 업데이트 기준: **2026-09-18**  
> 책의 학습 목표는 유지하면서 LangChain 1.x의 분리된 provider 패키지와 현재 메시지/스트리밍 API에 맞췄습니다. 모델 이름은 공급자 정책에 따라 바뀔 수 있으므로 환경 변수로 덮어쓸 수 있게 구성했습니다.


흐름은 동일합니다: Files API 업로드 → 처리 완료 대기 → 표준 LangChain 파일 content block으로 질문 → 파일 삭제. Legacy `google.generativeai` 대신 `google-genai` 클라이언트와 `langchain-google-genai` 4.x를 사용합니다.

`.env`에 `GOOGLE_API_KEY`를 설정하세요. 업로드된 파일에 접근할 수 있는 키이므로 노트북이나 저장소에 기록하지 않습니다.


In [ ]:
%pip install -qU 'langchain-google-genai>=4.4.0' google-genai python-dotenv requests


## 샘플 비디오 준비

이미 로컬 파일이 있으면 다운로드 셀을 건너뛰고 `video_path`만 바꾸세요.


In [ ]:
from pathlib import Path
import requests

video_path = Path("teddynote-sample-video.mp4")
sample_url = "https://www.dropbox.com/scl/fi/ugue14fyo010jgc7wuh4g/teddynote-sample-video.mp4?rlkey=wcsiktklt7jgoibsluft3m6z9&dl=1"

if not video_path.exists():
    with requests.get(sample_url, stream=True, timeout=60) as response:
        response.raise_for_status()
        with video_path.open("wb") as file:
            for chunk in response.iter_content(chunk_size=1024 * 1024):
                file.write(chunk)

print(video_path.resolve(), video_path.stat().st_size, "bytes")


## Files API 업로드와 처리 대기


In [ ]:
import time
from google import genai

client = genai.Client()
video_file = client.files.upload(file=video_path)
print("업로드 완료:", video_file.uri)

while video_file.state.name == "PROCESSING":
    print("비디오 처리 중...")
    time.sleep(5)
    video_file = client.files.get(name=video_file.name)

if video_file.state.name == "FAILED":
    raise RuntimeError(f"비디오 처리 실패: {video_file.state}")

print("사용 가능:", video_file.uri)


## LangChain 메시지로 질의


In [ ]:
import os
from dotenv import load_dotenv
from langchain.messages import HumanMessage
from langchain_google_genai import ChatGoogleGenerativeAI

load_dotenv()
model = ChatGoogleGenerativeAI(
    model=os.getenv("GEMINI_MODEL", "gemini-3.7-flash"),
    temperature=0,
    thinking_level="low",
)

summary_message = HumanMessage(
    content=[
        {"type": "text", "text": "이 영상을 짧게 요약해 주세요."},
        {
            "type": "file",
            "file_id": video_file.uri,
            "mime_type": "video/mp4",
        },
    ]
)
summary = model.invoke([summary_message])
print(summary.text)


## 스트리밍 질의


In [ ]:
timestamp_message = HumanMessage(
    content=[
        {
            "type": "text",
            "text": "Gencon을 언급한 시각과 그때 말한 내용을 알려 주세요.",
        },
        {
            "type": "file",
            "file_id": video_file.uri,
            "mime_type": "video/mp4",
        },
    ]
)

for chunk in model.stream([timestamp_message]):
    print(chunk.text, end="", flush=True)


## 업로드 파일 삭제

학습을 마친 뒤 명시적으로 삭제합니다. 실패한 실행에서도 정리하려면 실제 애플리케이션에서는 업로드 이후 구간을 `try/finally`로 감쌉니다.


In [ ]:
client.files.delete(name=video_file.name)
print("삭제 완료:", video_file.name)
